# Toy QeMCMC for Maximum Independent Set

This notebook is the readable reference solution for `QeMCMC-challenge.md`. It verifies exact optima, compares classical random-start MCMC, warm-started MCMC, and a toy Qiskit proposal kernel.

This implementation is a toy reproduction of the workflow structure, not a reproduction of the hardware-scale results and not a demonstration of quantum advantage.

## Environment and Imports

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np

from qemcmc_mis.classical_mcmc import greedy_independent_set_start, random_bitstring, run_classical_mcmc
from qemcmc_mis.experiments import run_repeated_experiments
from qemcmc_mis.graphs import get_problem_graphs
from qemcmc_mis.mis import all_bitstrings, brute_force_mis, independent_set_size, is_valid_independent_set
from qemcmc_mis.plotting import (
    plot_best_cost_convergence,
    plot_best_size_convergence,
    plot_energy_landscape_5,
    plot_final_cost_distribution,
    plot_graph,
    plot_sampled_bitstring_histogram,
    plot_sampled_cost_histogram,
    plot_success_rates,
    plot_valid_invalid_fraction,
)

try:
    from qemcmc_mis.quantum_proposal import make_simulator, sample_quantum_proposal
    HAS_QISKIT = True
except ImportError:
    HAS_QISKIT = False

## Graph Definitions

In [ ]:
graphs = get_problem_graphs()
assert graphs['graph_5'].number_of_nodes() == 5
assert graphs['graph_8'].number_of_nodes() == 8

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, graph) in zip(axes, graphs.items()):
    plot_graph(graph, name, ax=ax)
fig.tight_layout()

## Mathematical Model and Brute-Force Verification

The unconstrained objective is `C(x) = -sum_i x_i + penalty * sum_edges x_i x_j`. Main experiments use `penalty = n` so invalid high-cardinality states are safely unattractive.

In [ ]:
brute_results = {}
for name, graph in graphs.items():
    result = brute_force_mis(graph, penalty=graph.number_of_nodes())
    brute_results[name] = result
    expected_size = max(
        independent_set_size(x)
        for x in all_bitstrings(graph.number_of_nodes())
        if is_valid_independent_set(graph, x)
    )
    assert result['best_valid_size'] == expected_size
    print(name, 'MIS size:', result['best_valid_size'], 'optimal states:', result['optimal_valid_bitstrings'])

brute_results['graph_5']['states_df'].head(32)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_energy_landscape_5(brute_results['graph_5']['states_df'], '5-node energy landscape', ax=ax)
fig.tight_layout()

## Classical Random-Start and Warm-Started MCMC

In [ ]:
graph = graphs['graph_5']
rng = np.random.default_rng(7)
iterations = 200
beta = 1.5
penalty = graph.number_of_nodes()

random_trace = run_classical_mcmc(graph, random_bitstring(graph.number_of_nodes(), rng), iterations, beta, penalty, rng)
warm_start = greedy_independent_set_start(graph)
assert is_valid_independent_set(graph, warm_start)
warm_trace = run_classical_mcmc(graph, warm_start, iterations, beta, penalty, rng)

random_trace.tail(1), warm_trace.tail(1)

## Qiskit-Based Toy QeMCMC Proposal

The quantum circuit initializes the current state, applies RX mixers, optional edge-aware phase interactions, then measures. Qiskit measurement strings are reversed back into graph vertex order.

In [ ]:
if HAS_QISKIT:
    simulator = make_simulator()
    samples = [
        sample_quantum_proposal(graph, (0, 0, 0, 0, 0), simulator, 0.7, 0.4, rng)
        for _ in range(100)
    ]
    print('unique quantum proposals:', len(set(samples)))
    assert len(set(samples)) > 1
else:
    print('Qiskit is not installed in this kernel; skip quantum proposal cells until dependencies are synced.')

## Repeated Experiments

In [ ]:
methods = ['classical_random', 'classical_warm'] + (['qemcmc'] if HAS_QISKIT else [])
all_traces = []
all_summaries = []
for name, graph in graphs.items():
    traces, summary = run_repeated_experiments(
        graph,
        name,
        methods=methods,
        repetitions=5,
        iterations=50,
        beta=1.5,
        penalty=graph.number_of_nodes(),
        base_seed=2026,
    )
    all_traces.append(traces)
    all_summaries.append(summary)

import pandas as pd
traces_df = pd.concat(all_traces, ignore_index=True)
summary_df = pd.concat(all_summaries, ignore_index=True)
summary_df

## Result Plots

In [ ]:
for name, graph in graphs.items():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    plot_best_cost_convergence(traces_df, name, brute_results[name]['best_cost'], ax=axes[0])
    plot_best_size_convergence(traces_df, name, ax=axes[1])
    plot_final_cost_distribution(summary_df, name, ax=axes[2])
    fig.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_success_rates(summary_df, ax=axes[0])
plot_valid_invalid_fraction(summary_df, ax=axes[1])
fig.tight_layout()

one_trace = traces_df[(traces_df.graph_name == 'graph_5') & (traces_df.run_id == 0)]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_sampled_bitstring_histogram(one_trace, 'graph_5 sampled bitstrings', ax=axes[0])
plot_sampled_cost_histogram(one_trace, 'graph_5 sampled costs', ax=axes[1])
fig.tight_layout()

## Discussion and Limitations

The local classical proposal changes one bit at a time, so it explores the state graph by small Hamming-distance moves. The quantum proposal can jump to multiple-bit changes because the RX mixer and phase layer create a distribution over many measured bitstrings. On these tiny simulator instances, that different exploration behavior is the object of comparison; it may or may not improve success rate.

The warm start often begins at a valid, already-strong independent set, so it can look excellent on small graphs. That is useful as a baseline, not evidence that the quantum proposal is weak or strong in general.

This implementation is intentionally small: no hardware execution, no parallel tempering, no claim of quantum advantage, and no attempt to reproduce the scale or performance claims of the original QeMCMC paper.